# Evaluation
Prioritizing: 
1. Macro Pearson correlation
2. Macro RMSE
3. Group-level behavior
4. Consistency across seeds

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path('content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal')
RUNS_ROOT = ROOT / "runs"
OUTPUT_DIR = ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


RUNS = {
    "Flat linear": "flat_linear_ft_weighted",
    "Flat MLP": "flat_mlp_ft_weighted",
    "CPM parallel": "grouped_parallel_ft_weighted",
    "CPM sequential": "grouped_sequential_ft_weighted",
}


rows = []

for model_label, run_name in RUNS.items():
    run_dir = RUNS_ROOT / run_name

    with open(
        run_dir / "test_metrics.json",
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(file)

    group_metrics = metrics["group_metrics"]

    row = {
        "Model": model_label,
        "Run": run_name,
        "Macro RMSE": metrics["macro_rmse"],
        "Macro MAE": metrics["macro_mae"],
        "Macro Pearson": metrics["macro_pearson"],
    }

    for group_name in [
        "relevance",
        "implication",
        "coping",
        "normative",
    ]:
        row[f"{group_name.title()} RMSE"] = (
            group_metrics[group_name]["mean_rmse"]
        )

        row[f"{group_name.title()} Pearson"] = (
            group_metrics[group_name]["mean_pearson"]
        )

    rows.append(row)


comparison_df = pd.DataFrame(rows)

comparison_df.to_csv(
    OUTPUT_DIR / "architecture_comparison.csv",
    index=False,
)

print(comparison_df.round(4).to_string(index=False))

In [ ]:
comparison_df["Macro RMSE (1-5 scale)"] = (
    comparison_df["Macro RMSE"] * 4
)

comparison_df["Difference from paper"] = (
    comparison_df["Macro RMSE (1-5 scale)"] - 1.40
)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


RUN_DIR = Path(
    "/workspace/data/runs/grouped_parallel_ft_weighted"
)
OUTPUT_DIR = Path("/workspace/data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


PAPER_RMSE = {
    "suddenness": 1.33,
    "familiarity": 1.42,
    "predict_event": 1.47,
    "pleasantness": 1.30,
    "unpleasantness": 1.26,
    "goal_relevance": 1.57,
    "chance_responsblt": 1.43,
    "self_responsblt": 1.40,
    "other_responsblt": 1.57,
    "predict_conseq": 1.50,
    "goal_support": 1.33,
    "urgency": 1.43,
    "self_control": 1.35,
    "other_control": 1.36,
    "chance_control": 1.35,
    "accept_conseq": 1.36,
    "standards": 1.34,
    "social_norms": 1.44,
    "attention": 1.27,
    "not_consider": 1.53,
    "effort": 1.38,
}


with open(
    RUN_DIR / "test_metrics.json",
    "r",
    encoding="utf-8",
) as file:
    metrics = json.load(file)


rows = []

for dim_name, paper_rmse in PAPER_RMSE.items():
    rows.append({
        "Dimension": dim_name,
        "Your model": (
            metrics["per_dim_rmse"][dim_name] * 4
        ),
        "Original paper": paper_rmse,
    })

plot_df = pd.DataFrame(rows)

plot_df = plot_df.sort_values(
    "Your model",
    ascending=True,
).reset_index(drop=True)


y = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    plot_df["Original paper"],
    y,
    label="Original paper",
    marker="o",
)

ax.scatter(
    plot_df["Your model"],
    y,
    label="Your model",
    marker="x",
)

for idx, row in plot_df.iterrows():
    ax.plot(
        [row["Original paper"], row["Your model"]],
        [idx, idx],
        linewidth=0.8,
    )

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Dimension"])

ax.set_xlabel("RMSE on original 1–5 scale")
ax.set_ylabel("Appraisal dimension")
ax.set_title("Per-dimension RMSE comparison")

ax.legend()
ax.grid(axis="x", alpha=0.25)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "paper_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


RUN_DIR = Path(
    "/workspace/data/runs/grouped_parallel_ft_weighted"
)

log_df = pd.read_csv(RUN_DIR / "training_log.csv")

fig, ax = plt.subplots(figsize=(6.5, 4))

ax.plot(
    log_df["epoch"],
    log_df["train_objective_loss"],
    marker="o",
    label="Training",
)

ax.plot(
    log_df["epoch"],
    log_df["val_objective_loss"],
    marker="o",
    label="Validation",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Weighted MSE objective")
ax.set_title("Training and validation loss")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


RUN_DIR = Path(
    "/workspace/data/runs/grouped_parallel_ft_weighted"
)

batch_df = pd.read_csv(RUN_DIR / "batch_losses.csv")

batch_df["batch"] = range(1, len(batch_df) + 1)

window = 50

batch_df["smoothed_loss"] = (
    batch_df["batch_loss"]
    .rolling(window=window, min_periods=1)
    .mean()
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    batch_df["batch"],
    batch_df["batch_loss"],
    alpha=0.2,
    linewidth=0.7,
    label="Raw batch loss",
)

ax.plot(
    batch_df["batch"],
    batch_df["smoothed_loss"],
    linewidth=1.8,
    label=f"{window}-batch moving average",
)

ax.set_xlabel("Optimization batch")
ax.set_ylabel("Training objective")
ax.set_title("Batch-level training loss")
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()